# HLT Project — Human Deception Judgments

## Objective

The human experiment investigates how accurately people distinguish truthful from deceptive hotel reviews and which linguistic cues they report using when making their decisions.

A stratified subset of 120 reviews is selected from the corpus.

The subset is balanced across:

- gold labels;
- the five original hotel-based folds.

The 120 reviews are divided into six balanced blocks of 20 reviews, allowing each participant to evaluate only a manageable subset of the corpus.

Model predictions are not used during review selection, preventing the human comparison set from being intentionally enriched with easy or difficult examples for either automatic system.

Human judgments will later be compared with Qwen and RoBERTa predictions on exactly the same reviews.

## 1. Loadng the dataset

In [ ]:
from google.colab import drive
from pathlib import Path

import pandas as pd
import numpy as np


drive.mount("/content/drive")


PROJECT_DIR = Path(
    "/content/drive/MyDrive/HLT_project"
)

DATA_DIR = (
    PROJECT_DIR /
    "data" /
    "processed"
)

SAMPLES_DIR = (
    PROJECT_DIR /
    "samples"
)

RESULTS_DIR = (
    PROJECT_DIR /
    "results"
)


SAMPLES_DIR.mkdir(
    parents=True,
    exist_ok=True
)


df_clean = pd.read_csv(
    DATA_DIR /
    "positive_deceptive_reviews_clean.csv"
)


print(
    "Dataset shape:",
    df_clean.shape
)

Mounted at /content/drive
Dataset shape: (800, 8)


## 2. Select the Human Comparison Set

A stratified random sample of 120 reviews is selected independently of model predictions.

For each combination of hotel fold and gold label, 12 reviews are sampled:

5 folds × 2 labels × 12 reviews = 120 reviews.

This produces:

- 60 truthful reviews;
- 60 deceptive reviews;
- 24 reviews from each hotel fold.

In [ ]:
HUMAN_RANDOM_SEED = 123


human_sample_df = (
    df_clean
    .groupby(
        [
            "fold",
            "label"
        ],
        group_keys=False
    )
    .sample(
        n=12,
        random_state=HUMAN_RANDOM_SEED
    )
    .reset_index(drop=True)
)


print(
    "Human-study reviews:",
    len(human_sample_df)
)


print(
    "\nLabel distribution:"
)

print(
    human_sample_df[
        "label"
    ].value_counts()
)


print(
    "\nFold × label:"
)

display(
    pd.crosstab(
        human_sample_df["fold"],
        human_sample_df["label"]
    )
)

Human-study reviews: 120

Label distribution:
label
deceptive    60
truthful     60
Name: count, dtype: int64

Fold × label:


label,deceptive,truthful
fold,,
fold1,12,12
fold2,12,12
fold3,12,12
fold4,12,12
fold5,12,12


## 3. Create Six Balanced Human-Evaluation Blocks

The 120 reviews are divided into six blocks of 20 reviews.

Each block contains exactly:

- 10 truthful and 10 deceptive reviews;
- 4 reviews from each hotel fold;
- 2 truthful and 2 deceptive reviews from each fold.

This ensures that participants assigned to different blocks receive samples with the same class and fold composition.

In [ ]:
rng = np.random.default_rng(
    HUMAN_RANDOM_SEED
)


human_sample_df[
    "block"
] = None


for (
    fold,
    label
), group in human_sample_df.groupby(
    [
        "fold",
        "label"
    ]
):

    indices = group.index.to_numpy().copy()

    rng.shuffle(
        indices
    )


    block_assignments = np.repeat(
        np.arange(
            1,
            7
        ),
        2
    )


    for index, block_number in zip(
        indices,
        block_assignments
    ):

        human_sample_df.loc[
            index,
            "block"
        ] = f"B{block_number}"

In [ ]:
print(
    "Reviews per block:"
)

print(
    human_sample_df[
        "block"
    ].value_counts()
    .sort_index()
)


print(
    "\nBlock × label:"
)

display(
    pd.crosstab(
        human_sample_df["block"],
        human_sample_df["label"]
    )
)


print(
    "\nBlock × fold × label:"
)

display(
    pd.crosstab(
        [
            human_sample_df["block"],
            human_sample_df["fold"]
        ],
        human_sample_df["label"]
    )
)

Reviews per block:
block
B1    20
B2    20
B3    20
B4    20
B5    20
B6    20
Name: count, dtype: int64

Block × label:


label,deceptive,truthful
block,,
B1,10,10
B2,10,10
B3,10,10
B4,10,10
B5,10,10
B6,10,10



Block × fold × label:


label        deceptive  truthful
block fold                      
B1    fold1          2         2
      fold2          2         2
      fold3          2         2
      fold4          2         2
      fold5          2         2
B2    fold1          2         2
      fold2          2         2
      fold3          2         2
      fold4          2         2
      fold5          2         2
B3    fold1          2         2
      fold2          2         2
      fold3          2         2
      fold4          2         2
      fold5          2         2
B4    fold1          2         2
      fold2          2         2
      fold3          2         2
      fold4          2         2
      fold5          2         2
B5    fold1          2         2
      fold2          2         2
      fold3          2         2
      fold4          2         2
      fold5          2         2
B6    fold1          2         2
      fold2          2         2
      fold3          2         2
      fold4          2         2
      fold5          2         2

## 4. Randomize Review Order Within Each Human-Evaluation Block

The order of the 20 reviews within each block is randomized once using a fixed random seed.

The resulting order is frozen before data collection and is independent of review labels and model predictions.

In [ ]:
FORM_RANDOM_SEED = 456


form_blocks = {}


for block_name in sorted(
    human_sample_df["block"].unique()
):

    block_df = (
        human_sample_df[
            human_sample_df["block"]
            == block_name
        ]
        .sample(
            frac=1,
            random_state=FORM_RANDOM_SEED
        )
        .reset_index(drop=True)
        .copy()
    )


    block_df[
        "review_number"
    ] = np.arange(
        1,
        len(block_df) + 1
    )


    form_blocks[
        block_name
    ] = block_df

In [ ]:
for block_name, block_df in form_blocks.items():

    # Internal version
    internal_path = (
        SAMPLES_DIR /
        f"human_{block_name}_internal.csv"
    )

    block_df.to_csv(
        internal_path,
        index=False
    )


    # Form-building version
    form_path = (
        SAMPLES_DIR /
        f"human_{block_name}_form.csv"
    )


    block_df[
        [
            "review_number",
            "review_id",
            "text"
        ]
    ].to_csv(
        form_path,
        index=False
    )


print(
    "Six randomized human-study blocks saved."
)

Six randomized human-study blocks saved.


## Human Experiment Design — Final Setup

The human evaluation set contains 120 reviews selected independently of model predictions.

The sample is balanced across:

- 60 truthful and 60 deceptive reviews;
- five original hotel-based folds;
- six evaluation blocks of 20 reviews.

Each block contains:

- 10 truthful reviews;
- 10 deceptive reviews;
- 4 reviews from each fold.

The order of reviews within each block was randomized using a fixed random seed before data collection.

Participants evaluate one block of 20 reviews. For each review, they provide:

1. a TRUTHFUL / DECEPTIVE judgment;
2. a confidence rating from 1 to 5.

After completing all classifications, participants report the strategies and linguistic characteristics that influenced their judgments.

The class balance is not disclosed to participants.

The six blocks were converted into six separate Google Forms before data collection.

## Human Data Collection Closure

Human data collection was closed once all six experimental blocks had received at least five complete submissions.

Some blocks received additional responses before recruitment was closed. All complete valid responses are retained in the analysis; participants are not discarded merely to equalize the number of raters across blocks.

Consequently, the number of judgments per review may differ across blocks.

NB:The analysis protocol in the file HLT_Human_Analysis.ipynb was built in advance, before inspecting human performance.